# Telco Churn: Technical Exploratory Data Analysis (EDA)
**Objective:** To systematically map the mathematical drivers of customer churn across infrastructure, demographics, and financial variables using SQL Server.

### Methodology
To optimize query performance and avoid rewriting redundant `CASE WHEN` statements, I first established a session-level Temporary Table (`#churn_base`) that translates the binary 'Yes/No' churn string into an integer (`1/0`) for rapid aggregations.

```sql
-- MASTER TEMP TABLE: Build base dataset with integer churn flag for math
IF OBJECT_ID('tempdb..#churn_base') IS NOT NULL
DROP TABLE #churn_base;

SELECT
customerID,
Contract,
InternetService,
TechSupport,
Dependents,
SeniorCitizen,
StreamingTV,
CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END AS is_churned
INTO #churn_base
FROM
dbo.raw_telco_churn;

---
## Pillar 1: Core Infrastructure
**Hypothesis:** Is our flagship product (Fiber Optic) losing to competitors or legacy systems?

```sql
SELECT 
    InternetService,
    COUNT(customerID) AS total_customers,
    SUM(is_churned) AS churned_customers,
    CAST(SUM(is_churned) * 100.0 / COUNT(customerID) AS DECIMAL(5,2)) AS churn_rate_pct
FROM #churn_base
GROUP BY InternetService
ORDER BY churn_rate_pct DESC;

**Query Output:**
| InternetService | total_customers | churned_customers | churn_rate_pct |
| :--- | :--- | :--- | :--- |
| Fiber optic | 3096 | 1297 | 41.89 |
| DSL | 2421 | 459 | 18.96 |
| No | 1526 | 113 | 7.40 |

**Technical Insight:** A severe infrastructure failure is evident. The premium offering (Fiber Optic) is bleeding customers at ~42%, more than double the churn rate of the legacy DSL network.

---
## Pillar 2: The "Stickiness" Feature
**Hypothesis:** Does the security ecosystem (Tech Support) create customer loyalty?

```sql
SELECT 
    TechSupport,
    COUNT(customerID) AS total_customers,
    SUM(is_churned) AS churned_customers,
    CAST(SUM(is_churned) * 100.0 / COUNT(customerID) AS DECIMAL(5,2)) AS churn_rate_pct
FROM #churn_base
GROUP BY TechSupport
ORDER BY churn_rate_pct DESC;

**Query Output:**
| TechSupport | total_customers | churned_customers | churn_rate_pct |
| :--- | :--- | :--- | :--- |
| No | 3473 | 1446 | 41.64 |
| Yes | 2044 | 310 | 15.17 |
| No internet service | 1526 | 113 | 7.40 |

**Technical Insight:** Tech Support acts as a massive retention lever. Customers without it churn at ~42%, while those with it drop to a highly stable ~15%.

---
## Pillar 3: Demographics & Entertainment
**Hypothesis:** Are older demographics higher flight risks? Does bundling TV improve retention?

```sql
-- Testing Senior Citizens
SELECT 
    SeniorCitizen,
    COUNT(customerID) AS total_customers,
    SUM(is_churned) AS churned_customers,
    CAST(SUM(is_churned) * 100.0 / COUNT(customerID) AS DECIMAL(5,2)) AS churn_rate_pct
FROM #churn_base
GROUP BY SeniorCitizen
ORDER BY churn_rate_pct DESC; 

-- Testing Streaming TV
SELECT 
    StreamingTV,
    COUNT(customerID) AS total_customers,
    SUM(is_churned) AS churned_customers,
    CAST(SUM(is_churned) * 100.0 / COUNT(customerID) AS DECIMAL(5,2)) AS churn_rate_pct
FROM #churn_base
GROUP BY StreamingTV
ORDER BY churn_rate_pct DESC;

**Technical Insight:** 
1. **Demographics:** Senior Citizens (1) churn at an alarming 41.68% compared to non-seniors (0) at 23.61%. 
2. **Entertainment:** Streaming TV shows negligible retention value (No TV: 33.52% vs. Yes TV: 30.07%). It looks like statistical noise and not a core driver of loyalty.

---
## Pillar 4: Financials (Price Sensitivity)
**Hypothesis:** Are we pricing people out of our service?

```sql
SELECT 
    Churn,
    COUNT(customerID) AS total_customers,
    CAST(AVG(MonthlyCharges) AS DECIMAL(10,2)) AS avg_monthly_bill
FROM dbo.raw_telco_churn
GROUP BY Churn;

**Query Output:**
| Churn | total_customers | avg_monthly_bill |
| :--- | :--- | :--- |
| Yes | 1869 | 74.44 |
| No | 5174 | 61.27 |

**Technical Insight:** We are losing our highest-paying tier. Churned customers pay an average of $13 more per month than retained customers, indicating severe price-to-value dissatisfaction.

---
## Isolated CTE Analysis: Contracts & Cohorts
**Hypothesis:** Identify the financial bleed in short-term vs long-term plans, and map the customer survival curve to find onboarding failures.

```sql
-- CTE 1: Billing Contracts
WITH silver_churn_contract AS (
    SELECT customerID, Contract, CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END AS is_churned 
    FROM dbo.raw_telco_churn
)
SELECT Contract, COUNT(customerID) AS total_customers, SUM(is_churned) AS churned_customers, CAST((SUM(is_churned) * 100.0 ) / COUNT(customerID) AS DECIMAL(5,2)) AS churn_rate_pct
FROM silver_churn_contract
GROUP BY Contract
ORDER BY churn_rate_pct DESC;

-- CTE 2: Cohort Retention (Tenure)
WITH silver_churn_tenure AS (
    SELECT customerID, tenure, CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END AS is_churned, 
        CASE 
        WHEN tenure <= 6 THEN '0-6 Months'
        WHEN tenure > 6 AND tenure <= 12 THEN '6-12 Months'
        WHEN tenure > 12 AND tenure <= 24 THEN '1-2 Years'
        ELSE '2+ Years' 
    END AS tenure_bucket
    FROM dbo.raw_telco_churn
)
SELECT tenure_bucket, COUNT(customerID) AS total_customers, SUM(is_churned) AS churned_customers, CAST((SUM(is_churned) * 100.0 ) / COUNT(customerID) AS DECIMAL(5,2))  AS churn_rate_pct
FROM silver_churn_tenure
GROUP BY tenure_bucket
ORDER BY churn_rate_pct DESC;

**Technical Insight:** 
* **Contracts:** Month-to-month contracts are bleeding out at 42.71%, while 1-year and 2-year contracts drop to 11.27% and 2.83% respectively. 
* **Tenure:** Over half (52.94%) of all churn happens within the first 6 months. This points to a massive failure in the onboarding process, rather than long-term fatigue.

---
## Deep Dive: Cross-Sectional Analysis & The "Doomsday" Segment
**Hypothesis:** Can Tech Support save our failing Fiber Optic product? What is the absolute worst-case intersection of variables?

```sql
-- DEEP DIVE 1: Product vs. Ecosystem
SELECT 
    InternetService, TechSupport, COUNT(customerID) AS total_customers, SUM(is_churned) AS churned_customers, CAST(SUM(is_churned) * 100.0 / COUNT(customerID) AS DECIMAL(5,2)) AS churn_rate_pct
FROM #churn_base
WHERE InternetService = 'Fiber optic'
GROUP BY InternetService, TechSupport
ORDER BY churn_rate_pct DESC;

-- DEEP DIVE 2: The Doomsday Cohort
SELECT 
    Contract, InternetService, SeniorCitizen, COUNT(customerID) AS total_customers, SUM(is_churned) AS churned_customers, CAST(SUM(is_churned) * 100.0 / COUNT(customerID) AS DECIMAL(5,2)) AS churn_rate_pct
FROM #churn_base
WHERE InternetService = 'Fiber optic' AND SeniorCitizen = 1
GROUP BY Contract, InternetService, SeniorCitizen
ORDER BY churn_rate_pct DESC;

**Technical Insight:** 
1. **The Savior:** Tech Support cuts Fiber Optic churn in half (from 49.37% down to 22.63%).
2. **The Doomsday Segment:** The exact mathematical intersection of failure is a Senior Citizen on a Month-to-Month Fiber Optic plan. This isolated cohort has an astonishing churn rate of **57.86%**, making it the most critical liability in the dataset.